# Atualizador do Painel — Ciências Humanas

Este notebook mantém o painel de estudos em dia **sem que você precise editar nada à mão**. Ele lê o repositório inteiro e devolve quatro arquivos:

1. **`README.md`** atualizado — com a tabela "Painel de temas" regenerada (contagens por tema).
2. **`indices/indice.json`** — o mapa completo do acervo: livros, fichas, conceitos e trechos.
3. **`flashcards.csv`** — pronto para importar no Anki (revisão espaçada dos seus conceitos).
4. **`relatorio-problemas.md`** — auditoria do repositório: links quebrados, wikilinks antigos que não funcionam no GitHub, conceitos possivelmente duplicados e notas sem definição.

**Como usar (não precisa programar):**

1. No GitHub, na página do repositório, clique no botão verde **Code** e depois em **Download ZIP**.
2. Aqui no Colab, execute a Célula 1 e envie esse zip.
3. Clique em **Ambiente de execução > Executar tudo** (ou execute as células em ordem, uma por uma).
4. No final, o zip de atualização baixa sozinho: descompacte e arraste o conteúdo de volta para o repositório (os arquivos de mesmo nome substituem os antigos).

Recomendação: rode uma vez por mês, ou sempre que acumular um lote de notas novas.

In [ ]:
# CELULA 1 — Enviar o zip do repositório (baixado do GitHub)
from google.colab import files
import zipfile, pathlib, shutil

print('No GitHub, clique no botão verde Code > Download ZIP.')
print('Depois selecione aqui o arquivo zip baixado.')
enviados = files.upload()
nome_zip = list(enviados.keys())[0]

base = pathlib.Path('/content/repo_extracao')
if base.exists():
    shutil.rmtree(base)
base.mkdir()
with zipfile.ZipFile(nome_zip) as z:
    z.extractall(base)

subpastas = [p for p in base.iterdir() if p.is_dir()]
pasta_repo = subpastas[0] if len(subpastas) == 1 else base
print('Repositório extraído em:', pasta_repo)

mds = sorted(pasta_repo.rglob('*.md'))
print(f'{len(mds)} arquivos .md encontrados.')
if not (pasta_repo / 'conceitos').exists():
    print('Aviso: não encontrei a pasta conceitos/ — este zip é do repositório do painel?')

## Leitura do repositório

A célula abaixo lê as notas de `conceitos/`, `trechos/` e `acervo/`, incluindo o cabeçalho de dados de cada nota (as linhas entre `---` no começo do arquivo). Não altera nada — só lê e organiza na memória para as próximas etapas.

In [ ]:
# CELULA 2 — Ler o repositório (notas e cabeçalhos de dados)
import re, unicodedata, json, csv, pathlib, difflib, datetime

def parse_frontmatter(texto):
    """Lê as linhas entre --- no começo do arquivo (dados: tema, obra, tags...)."""
    dados = {}
    t = texto.lstrip()
    if not t.startswith('---'):
        return dados
    partes = t.split('---', 2)
    if len(partes) < 3:
        return dados
    for linha in partes[1].splitlines():
        if ':' not in linha:
            continue
        chave, valor = linha.split(':', 1)
        dados[chave.strip().lower()] = valor.strip().strip('"').strip("'")
    return dados

def parse_tags(valor):
    valor = (valor or '').strip().strip('[]')
    return [t.strip() for t in valor.split(',') if t.strip()]

def extrair_secao(texto, titulo):
    """Pega o texto de uma seção (## Título) até a próxima seção."""
    padrao = r'^##\s+' + re.escape(titulo) + r'\s*$'
    m = re.search(padrao, texto, flags=re.M | re.I)
    if not m:
        return ''
    bloco = texto[m.end():].split('\n## ', 1)[0]
    linhas = []
    for linha in bloco.splitlines():
        l = linha.strip()
        if not l:
            if linhas:
                break
            continue
        if l.startswith('<!--') or l.startswith('>'):
            continue
        linhas.append(l)
    return ' '.join(linhas).strip()

def titulo_do_arquivo(texto):
    m = re.search(r'^#\s+(.+)$', texto, flags=re.M)
    return m.group(1).strip() if m else ''

registros = {'conceitos': [], 'trechos': [], 'livros': [], 'fichas': []}

if (pasta_repo / 'conceitos').exists():
    for arq in sorted((pasta_repo / 'conceitos').glob('*.md')):
        if arq.name.upper() == 'README.MD':
            continue
        texto = arq.read_text(encoding='utf-8')
        fm = parse_frontmatter(texto)
        conceito = fm.get('conceito') or fm.get('eixo') or titulo_do_arquivo(texto).replace('Conceito —', '').replace('Conceito -', '').replace('Eixo —', '').replace('Eixo -', '').strip()
        registros['conceitos'].append({
            'arquivo': arq.name,
            'conceito': conceito,
            'tema': (fm.get('tema') or 'sem-tema').strip().lower(),
            'obra_origem': fm.get('obra_origem', ''),
            'autor_origem': fm.get('autor_origem', ''),
            'definicao': extrair_secao(texto, 'Definição em uma frase') or extrair_secao(texto, 'A questão em uma frase'),
            'tags': parse_tags(fm.get('tags', '')),
        })

if (pasta_repo / 'trechos').exists():
    for arq in sorted((pasta_repo / 'trechos').glob('*.md')):
        if arq.name.upper() == 'README.MD':
            continue
        texto = arq.read_text(encoding='utf-8')
        fm = parse_frontmatter(texto)
        registros['trechos'].append({
            'arquivo': arq.name,
            'titulo': titulo_do_arquivo(texto),
            'tema': (fm.get('tema') or 'sem-tema').strip().lower(),
            'obra': fm.get('obra', ''),
            'autor': fm.get('autor', ''),
            'capitulo': fm.get('capitulo', ''),
            'pagina': fm.get('pagina', ''),
        })

if (pasta_repo / 'acervo').exists():
    for arq in sorted((pasta_repo / 'acervo').rglob('*.md')):
        if arq.name.upper() == 'README.MD':
            continue
        texto = arq.read_text(encoding='utf-8')
        fm = parse_frontmatter(texto)
        reg = {'arquivo': arq.name,
               'tema': (fm.get('tema') or 'sem-tema').strip().lower(),
               'palavras': len(texto.split())}
        if arq.name.lower().endswith('-ficha.md'):
            registros['fichas'].append(reg)
        else:
            registros['livros'].append(reg)

# o tema dos livros convertidos pode estar no indice.json antigo
indice_antigo = pasta_repo / 'indices' / 'indice.json'
if indice_antigo.exists():
    try:
        antigo = json.loads(indice_antigo.read_text(encoding='utf-8'))
        mapa = {l.get('arquivo'): (l.get('tema') or 'sem-tema') for l in antigo.get('livros', [])}
        for r in registros['livros']:
            if r['tema'] == 'sem-tema' and r['arquivo'] in mapa:
                r['tema'] = mapa[r['arquivo']]
    except Exception as erro:
        print('Aviso: não consegui ler o indice.json antigo:', erro)

print(f"Conceitos: {len(registros['conceitos'])} | Trechos: {len(registros['trechos'])} | Livros: {len(registros['livros'])} | Fichas: {len(registros['fichas'])}")

## Geração do índice e da tabela de temas

As próximas duas células criam o `indice.json` completo e o `README.md` com a tabela "Painel de temas" regenerada. Se o seu README ainda não tiver as marcações de regeneração, a tabela sai em um arquivo separado para você colar.

In [ ]:
# CELULA 3 — Regenerar o indice.json
saida = pathlib.Path('/content/atualizacao')
(saida / 'indices').mkdir(parents=True, exist_ok=True)

indice = {
    'projeto': 'painel-ciencias-humanas',
    'descricao': 'Mapa geral do acervo processado. Gerado automaticamente pelo notebook atualizador-do-painel.',
    'atualizado_em': datetime.datetime.now().isoformat(timespec='seconds'),
    'livros': registros['livros'],
    'fichas': registros['fichas'],
    'conceitos': registros['conceitos'],
    'trechos': registros['trechos'],
}
(saida / 'indices' / 'indice.json').write_text(json.dumps(indice, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"indice.json regenerado: {len(registros['livros'])} livro(s), {len(registros['conceitos'])} conceito(s), {len(registros['trechos'])} trecho(s).")

In [ ]:
# CELULA 4 — Regenerar a tabela do Painel de temas no README
temas_fixos = [('historia', 'História'), ('sociologia', 'Sociologia'), ('filosofia', 'Filosofia'),
               ('ciencia-politica', 'Ciência Política'), ('antropologia', 'Antropologia'),
               ('economia', 'Economia'), ('geografia', 'Geografia'), ('psicologia', 'Psicologia'),
               ('psicologia-social', 'Psicologia Social')]

def contar(lista):
    c = {}
    for r in lista:
        chave = r.get('tema', 'sem-tema')
        c[chave] = c.get(chave, 0) + 1
    return c

conceitos_tema = contar(registros['conceitos'])
livros_tema = contar(registros['livros'])
fichas_tema = contar(registros['fichas'])
for chave, n in fichas_tema.items():
    livros_tema[chave] = livros_tema.get(chave, 0) + n

todos = dict(temas_fixos)
for chave in set(conceitos_tema) | set(livros_tema):
    if chave not in todos:
        todos[chave] = chave.replace('-', ' ').title()

linhas = ['| Tema | Status | Notas de conceito | Livros processados |', '|---|---|---|---|']
for chave, nome in todos.items():
    nc = conceitos_tema.get(chave, 0)
    nl = livros_tema.get(chave, 0)
    status = 'Ativo' if (nc + nl) > 0 else 'Em breve'
    linhas.append(f'| {nome} | {status} | {nc} | {nl} |')
tabela = '\n'.join(linhas)

inicio_marcador = '<!-- PAINEL-TEMAS-INICIO -->'
fim_marcador = '<!-- PAINEL-TEMAS-FIM -->'
readme_path = pasta_repo / 'README.md'
readme_original = readme_path.read_text(encoding='utf-8') if readme_path.exists() else ''

if inicio_marcador in readme_original and fim_marcador in readme_original:
    pre = readme_original.split(inicio_marcador)[0]
    pos = readme_original.split(fim_marcador)[1]
    readme_novo = pre + inicio_marcador + '\n\n' + tabela + '\n\n' + fim_marcador + pos
    (saida / 'README.md').write_text(readme_novo, encoding='utf-8')
    print('README.md regenerado com a tabela de temas atualizada.')
else:
    (saida / 'tabela-temas.md').write_text('## Painel de temas\n\n' + tabela + '\n', encoding='utf-8')
    print('ATENÇÃO: o README não tem as marcações de regeneração.')
    print('A tabela foi salva em tabela-temas.md — cole-a no lugar da tabela antiga.')
print()
print(tabela)

## Auditoria do repositório

A célula abaixo procura quatro tipos de problema e guarda tudo no `relatorio-problemas.md`:

- **Links quebrados** (apontando para arquivos que não existem);
- **Wikilinks antigos** (`[[assim]]`), que não viram link no GitHub;
- **Conceitos duplicados ou parecidos** (ex.: "habitus" e "habitos");
- **Notas de conceito sem definição** — que produziriam flashcards vazios no Anki.

A auditoria cobre as suas notas (`conceitos/`, `trechos/` e `acervo/`). Os exemplos dos templates e dos guias não entram na conta — eles têm links fictícios de propósito.

In [ ]:
# CELULA 5 — Auditoria: links, duplicados e notas incompletas
problemas = []

# a auditoria cobre apenas as suas notas; templates e guias não entram (têm exemplos de propósito)
arquivos_auditar = []
for pasta in ['conceitos', 'trechos', 'acervo']:
    pasta_caminho = pasta_repo / pasta
    if pasta_caminho.exists():
        arquivos_auditar.extend(sorted(pasta_caminho.rglob('*.md')))
arquivos_auditar = [a for a in arquivos_auditar if a.name.upper() != 'README.MD']

for arq in arquivos_auditar:
    texto = arq.read_text(encoding='utf-8')
    relativo = arq.relative_to(pasta_repo)
    for alvo in re.findall(r'\]\(([^)]+)\)', texto):
        alvo = alvo.split('#')[0].strip()
        if not alvo or alvo.startswith('http') or not alvo.lower().endswith('.md'):
            continue
        if (arq.parent / alvo).exists() or (pasta_repo / alvo).exists():
            continue
        problemas.append(f'- Link quebrado em {relativo}: ({alvo})')
    for wik in re.findall(r'\[\[[^\]]+\]\]', texto):
        problemas.append(f'- Wikilink [[...]] em {relativo} (não vira link no GitHub): {wik}')

def normalizar(s):
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+', ' ', s).strip()

nomes = [(r['arquivo'], normalizar(r['conceito'])) for r in registros['conceitos'] if r['conceito']]
for i in range(len(nomes)):
    for j in range(i + 1, len(nomes)):
        arq_a, na = nomes[i]
        arq_b, nb = nomes[j]
        if not na or not nb:
            continue
        conceito_a = registros['conceitos'][i]['conceito']
        conceito_b = registros['conceitos'][j]['conceito']
        if na == nb:
            problemas.append(f'- Duplicado provável: conceitos/{arq_a} e conceitos/{arq_b} tratam do mesmo conceito ("{conceito_a}"). Considere juntar as duas notas.')
        elif difflib.SequenceMatcher(None, na, nb).ratio() >= 0.85:
            problemas.append(f'- Conceitos parecidos: "{conceito_a}" ({arq_a}) e "{conceito_b}" ({arq_b}). Confira se são o mesmo conceito.')

for r in registros['conceitos']:
    if not r['definicao']:
        problemas.append(f"- Nota sem definição: conceitos/{r['arquivo']} — o flashcard correspondente ficará de fora; complete a seção 'Definição em uma frase'.")

cabecalho = ['# Relatório de problemas do painel',
             'Gerado em ' + datetime.datetime.now().strftime('%d/%m/%Y %H:%M'),
             '']
if problemas:
    cabecalho.append(f'**{len(problemas)} ponto(s) para conferir:**')
    cabecalho.extend(problemas)
else:
    cabecalho.append('Nenhum problema encontrado. Painel limpo.')

(saida / 'relatorio-problemas.md').write_text('\n'.join(cabecalho), encoding='utf-8')
print('\n'.join(cabecalho))

In [ ]:
# CELULA 6 — Gerar o flashcards.csv para o Anki
total_cartoes = 0
with (saida / 'flashcards.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['Frente', 'Verso', 'Tags'])
    for r in registros['conceitos']:
        if not r['definicao']:
            continue
        frente = 'O que é ' + r['conceito'] + '?'
        verso = r['definicao']
        if r['obra_origem']:
            verso += ' — Fonte: ' + r['obra_origem']
        w.writerow([frente, verso, ' '.join(r['tags'][:5])])
        total_cartoes += 1

print(f'flashcards.csv criado com {total_cartoes} cartão(ões).')
if total_cartoes == 0:
    print('Nenhum cartão: ainda não há conceitos com definição preenchida. Crie as primeiras notas e rode de novo.')

## Baixar a atualização

A última célula empacota os quatro arquivos e inicia o download. Depois:

1. Descompacte o zip baixado.
2. No GitHub: **Add file > Upload files** e arraste o conteúdo — o `README.md` e o `indice.json` substituem os antigos automaticamente.
3. Importe o `flashcards.csv` no Anki (passo a passo no Guia 02, seção de retenção).
4. Leia o `relatorio-problemas.md` e corrija o que ele aponta (ele é só para leitura, não precisa subir).

In [ ]:
# CELULA 7 — Empacotar e baixar
import shutil
from google.colab import files as downloader

caminho_zip = shutil.make_archive('/content/atualizacao-painel', 'zip', '/content/atualizacao')
downloader.download(caminho_zip)
print('Download iniciado. Resumo:')
print(' - README.md              -> substitui o atual (tabela de temas nova)')
print(' - indices/indice.json    -> substitui o atual (mapa completo)')
print(' - flashcards.csv         -> importe no Anki')
print(' - relatorio-problemas.md -> leia e corrija o que apontar')